In [3]:
import re 
import numpy as np 
import pandas as pd 
from pathlib import Path 

In [4]:
raiz = Path.cwd()

while not (raiz / ".venv").exists() and raiz != raiz.parent:
    raiz = raiz.parent

carpeta_raw = raiz / "Data" / "raw"
carpeta_processed = raiz / "Data"/ "processed"

print( "Raíz del proyecto:", raiz)
print( "Archivo en raw:", [f.name for f in carpeta_raw.glob("*.xls*")])
    

Raíz del proyecto: /Users/admin/Desktop/ipvn-colombia
Archivo en raw: ['anex-IPVN-VarTrimTotalDestinoViv-IItrim2026.xlsx']


In [5]:
Archivo = carpeta_raw / "anex-IPVN-VarTrimTotalDestinoViv-IItrim2026.xlsx"

xls = pd.ExcelFile(Archivo)
print(xls.sheet_names)


['Trimestral']


In [6]:
Sheet = "Trimestral"

raw = pd.read_excel(Archivo, sheet_name=Sheet, header= None)
print("Tamaño de la sheet:", raw.shape)

Tamaño de la sheet: (66, 39)


In [7]:
texto = raw.astype(str).apply(lambda col: col.str.strip())
filas, columnas = np.where(texto == "Año")
fila_enc = filas[0]
print("Fila del encabezado principal:", fila_enc)

raw.iloc[fila_enc:fila_enc + 4, :8]

Fila del encabezado principal: 8


,0,1,2,3,4,5,6,7
8,Año,Trimestre,Total,Total Apartamentos,Total \nCasas,Armenia AU,NaN,Barranquilla AU
9,NaN,NaN,NaN,NaN,NaN,Apartamentos,Casas,Apartamentos
10,2015,I,2.37,2.36,2.49,-2.52,1.99,3.68
11,NaN,II,1.37,1.39,1.15,3.52,2.16,1.69


In [8]:
def limpiar_texto(x):
    """Normaliza un texto de encabezado."""
    if pd.isna(x):
        return np.nan
    x = str(x).replace("\\n", " ")        
    x = " ".join(x.split())               
    x = re.sub(r"\d+$", "", x).strip()  
    return x

In [9]:
tabla = raw.iloc[fila_enc:].dropna(axis=1, how="all")

nivel1 = tabla.iloc[0].map(limpiar_texto).ffill() 
nivel2 = tabla.iloc[1].map(limpiar_texto)           

nombres = []
for n1, n2 in zip(nivel1, nivel2):
    if pd.isna(n1):
        nombres.append(np.nan)
    elif pd.isna(n2):
        nombres.append(n1)
    else:
        nombres.append(f"{n1}|{n2}")

print(nombres)

['Año', 'Trimestre', 'Total', 'Total Apartamentos', 'Total Casas', 'Armenia AU|Apartamentos', 'Armenia AU|Casas', 'Barranquilla AU|Apartamentos', 'Barranquilla AU|Casas', 'Bogotá+Soacha|Apartamentos', 'Bogotá+Soacha|Casas', 'Bucaramanga AM|Apartamentos', 'Bucaramanga AM|Casas', 'Cali AU|Apartamentos', 'Cali AU|Casas', 'Medellín AM|Apartamentos', 'Medellín AM|Casas', 'Pereira AU|Apartamentos', 'Pereira AU|Casas', 'Bogotá+Cundinamarca|Apartamentos', 'Bogotá+Cundinamarca|Casas', 'Cartagena AU|Apartamentos', 'Cartagena AU|Casas', 'Manizales AU|Apartamentos', 'Manizales AU|Casas', 'Popayán AU|Apartamentos', 'Popayán AU|Casas', 'Cundinamarca|Apartamentos', 'Cundinamarca|Casas', 'Neiva AU|Apartamentos', 'Neiva AU|Casas', 'Villavicencio AU|Apartamentos', 'Villavicencio AU|Casas', 'Pasto AU|Apartamentos', 'Pasto AU|Casas', 'Cúcuta AM|Apartamentos', 'Cúcuta AM|Casas', 'Ibagué AU|Apartamentos', 'Ibagué AU|Casas']


In [10]:
ancho = tabla.iloc[2:].copy()
ancho.columns = nombres
ancho = ancho.loc[:, ancho.columns.notna()]  

ancho = ancho.rename(columns={
    "Año": "anio",
    "Trimestre": "trimestre",
    "Total": "Total nacional|Total",
    "Total Apartamentos": "Total nacional|Apartamentos",
    "Total Casas": "Total nacional|Casas",
})


ancho["trimestre"] = ancho["trimestre"].astype(str).str.strip()
ancho = ancho[ancho["trimestre"].isin(["I", "II", "III", "IV"])].copy()

ancho["anio"] = pd.to_numeric(ancho["anio"].ffill()).astype(int)

ancho = ancho[ancho["anio"] >= 2015].copy()

ancho[["anio", "trimestre"]].head(8)

,anio,trimestre
10,2015,I
11,2015,II
12,2015,III
13,2015,IV
14,2016,I
15,2016,II
16,2016,III
17,2016,IV


In [15]:
romanos = {"I": 1, "II": 2, "III": 3, "IV": 4}

# Convertir solo si todavía están en números romanos
if not pd.api.types.is_numeric_dtype(ancho["trimestre"]):
    ancho["trimestre"] = ancho["trimestre"].map(romanos)

assert ancho["trimestre"].notna().all(), "Hay trimestres que no se pudieron convertir"

ancho["fecha"] = pd.to_datetime(pd.DataFrame({
    "year": ancho["anio"],
    "month": (ancho["trimestre"] - 1) * 3 + 1,
    "day": 1,
}))

ancho["periodo"] = ancho["anio"].astype(str) + "-T" + ancho["trimestre"].astype(str)

ancho[["anio", "trimestre", "periodo", "fecha"]].head(5)

,anio,trimestre,periodo,fecha
10,2015,1,2015-T1,2015-01-01
11,2015,2,2015-T2,2015-04-01
12,2015,3,2015-T3,2015-07-01
13,2015,4,2015-T4,2015-10-01
14,2016,1,2016-T1,2016-01-01


In [16]:
cols_valor = [c for c in ancho.columns if "|" in str(c)]

# Contar cuántos "(-)" hay antes de reemplazarlos
es_guion = ancho[cols_valor].astype(str).apply(lambda s: s.str.strip()) == "(-)"
n_guiones = es_guion.sum().sum()

ancho[cols_valor] = ancho[cols_valor].mask(es_guion, np.nan)

# Convertir a número y detectar textos inesperados
convertidos = ancho[cols_valor].apply(pd.to_numeric, errors="coerce")
problemas = ancho[cols_valor].notna() & convertidos.isna()

print("Columnas de valores:", len(cols_valor))
print("Celdas '(-)' convertidas a nulo:", n_guiones)
print("Valores no numéricos inesperados:", problemas.sum().sum())

ancho[cols_valor] = convertidos

Columnas de valores: 37
Celdas '(-)' convertidas a nulo: 0
Valores no numéricos inesperados: 0


In [17]:
print("Filas:", len(ancho), "| Años:", ancho["anio"].min(), "-", ancho["anio"].max())

assert len(ancho) == 46, "Deberían ser 46 trimestres (2015-T1 a 2026-T2)"
assert not ancho.duplicated(["anio", "trimestre"]).any(), "Hay trimestres duplicados"

def valor(anio, trim, col):
    return ancho.loc[(ancho["anio"] == anio) & (ancho["trimestre"] == trim), col].iloc[0]

# Comparación contra valores visibles en el Excel
print("2015-T1 Total nacional:", round(valor(2015, 1, "Total nacional|Total"), 2), "(esperado 2.37)")
print("2015-T1 Medellín Aptos:", round(valor(2015, 1, "Medellín AM|Apartamentos"), 2), "(esperado 3.03)")
print("2026-T2 Total Casas:  ", round(valor(2026, 2, "Total nacional|Casas"), 2), "(esperado 3.68)")

Filas: 46 | Años: 2015 - 2026
2015-T1 Total nacional: 2.37 (esperado 2.37)
2015-T1 Medellín Aptos: 3.03 (esperado 3.03)
2026-T2 Total Casas:   3.68 (esperado 3.68)


In [18]:
largo = ancho.melt(
    id_vars=["anio", "trimestre", "periodo", "fecha"],
    value_vars=cols_valor,
    var_name="columna",
    value_name="variacion_trimestral",
)

largo[["area", "tipo_vivienda"]] = largo["columna"].str.split("|", expand=True, regex=False)
largo = largo.drop(columns="columna")

# Nombre de ciudad sin "AU"/"AM" (útil para mapas en Tableau)
largo["ciudad"] = largo["area"].str.replace(r"\s+(AU|AM)$", "", regex=True)

# Tipo de agregación, para no mezclar áreas que se solapan
largo = largo[~largo["area"].isin(["Bogotá+Cundinamarca", "Cundinamarca"])].copy()
largo["ciudad"] = largo["ciudad"].replace({"Bogotá+Soacha": "Bogotá"})

largo = largo[["anio", "periodo", "fecha", "ciudad", "tipo_vivienda", "variacion_trimestral"]]
largo.head(10)


,anio,periodo,fecha,ciudad,tipo_vivienda,variacion_trimestral
0,2015,2015-T1,2015-01-01,Total nacional,Total,2.37
1,2015,2015-T2,2015-04-01,Total nacional,Total,1.37
2,2015,2015-T3,2015-07-01,Total nacional,Total,1.46
3,2015,2015-T4,2015-10-01,Total nacional,Total,1.52
4,2016,2016-T1,2016-01-01,Total nacional,Total,2.31
5,2016,2016-T2,2016-04-01,Total nacional,Total,2.23
6,2016,2016-T3,2016-07-01,Total nacional,Total,2.16
7,2016,2016-T4,2016-10-01,Total nacional,Total,1.37
8,2017,2017-T1,2017-01-01,Total nacional,Total,1.81
9,2017,2017-T2,2017-04-01,Total nacional,Total,1.20


In [19]:
print(tabla_processed := carpeta_processed / "ipvn_trimestral.csv")

/Users/admin/Desktop/ipvn-colombia/Data/processed/ipvn_trimestral.csv


In [22]:
salida = carpeta_processed / "ipvn_variacion_trimestral_2015_2026.csv"

(largo.sort_values(["ciudad", "tipo_vivienda", "fecha"])
      .to_csv(salida, index=False, encoding="utf-8-sig"))

print("Guardado en:", salida)  

Guardado en: /Users/admin/Desktop/ipvn-colombia/Data/processed/ipvn_variacion_trimestral_2015_2026.csv
